## Part 1: Clustering

### readvectorseq

In [3]:
import numpy as np
import re
import random

In [4]:
# This function reads the dataset file and converts each row into a vector
# Input: filename (path to dataset)
# Output: list of numpy arrays (points)

def readVectorsSeq(file_path):
    data = []  # storing all points

    with open(file_path, 'r') as f:
        for line in f:
            vals = line.strip().split(',')  # split w.r.t comma
            vals = list(map(float, vals)) # convert to float

            arr = np.array(vals)
            data.append(arr)

    return data

In [5]:
P = readVectorsSeq("Assignment 4- datasets\\Q1- UCI Spam clustering\\spambase.data")
print(len(P))
print(len(P[0]))

4601
58


In [6]:
# making the functiions RDD compatible
def ensure_list(P):
    try:
        return P.collect()   # if RDD
    except:
        return P

### kcenter

In [7]:
# Idea: pick points that are farthest from existing centers
# The distance function (Euclidean distance)
def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

# Input: P (points), k (number of centers)
# Output: list of k centers
def kcenter(P, k):
    P = ensure_list(P)
    
    centers = [P[0]]
    # pick first center randomly
    first_idx = np.random.randint(len(P))
    centers = [P[first_idx]]

    # initial distances
    dists = []
    for p in P:
        d = euclidean_distance(p, centers[0])
        dists.append(d)

    # Step 3: pick remaining centers
    for _ in range(1, k):
        # pick point with maximum distance
        next_center = P[np.argmax(dists)]
        centers.append(next_center)

        # update distances
        for i, p in enumerate(P):
            dist = euclidean_distance(p, next_center)
            dists[i] = min(dists[i], dist)

    return centers

In [8]:
k = 10
C = kcenter(P, k)

print("Number of centers:", len(C))

Number of centers: 10


### KMeansPP

In [9]:
# Implements k-means++ initialization
# Idea: choose next center based on probability (farther points more likely)
# Input: P (points), k
# Output: list of centers

def kmeansPP(P, k):
    P = ensure_list(P)
    n = len(P)

    # pick first center randomly
    centers = [P[random.randint(0, n - 1)]]

    # initialize distances
    dists = []
    for p in P:
        d = euclidean_distance(p, centers[0])
        dists.append(d * d)  # squared

    for _ in range(1, k):
        # compute probability distribution
        total = 0
        for d in dists:
            total += d
        probs = [d / total for d in dists]

        # pick next center based on probability
        r = random.random()
        cumulative = 0

        for i, p in enumerate(P):
            cumulative += probs[i]
            if r <= cumulative:
                centers.append(p)
                break

        # update distances
        for i, p in enumerate(P):
            dist = euclidean_distance(p, centers[-1])**2
            dists[i] = min(dists[i], dist)

    return centers

In [10]:
k = 10
C = kmeansPP(P, k)

print("Centers:", len(C))

Centers: 10


### KmeansObj

Objective = (1 / |P|) * Σ min distance²(p, c)

In [11]:
# Computes clustering objective (average squared distance)
# Input: P (points), C (centers)
# Output: average squared distance

def kmeansObj(P, C):
    P = ensure_list(P)
    C = ensure_list(C)
    total = 0

    for pt in P:
        best = float('inf')

        for center in C:
            d = euclidean_distance(pt, center)
            d = d * d  # squaring

            if d < best:
                best = d

        total += best
    # return average value
    return total / len(P)

In [12]:
C = kmeansPP(P, 10)
obj = kmeansObj(P, C)

print("Objective value:", obj)

Objective value: 37529.154235651244


In [13]:
#verification

C1 = kcenter(P, 10)
obj1 = kmeansObj(P, C1)

C2 = kmeansPP(P, 10)
obj2 = kmeansObj(P, C2)

print("kcenter objective:", obj1)
print("kmeans++ objective:", obj2)
if obj1 < obj2:
    print("kcenter obj<kmeans++ obj")
else:
    print("kmeans++ obj<kcenter obj") #intended result: kmeans++ should be better than kcenter
# k-center spreads points (max distance)
#k-means++ minimizes clustering cost

kcenter objective: 83410.29208275283
kmeans++ objective: 23619.504040255866
kmeans++ obj<kcenter obj


### Results & Analysis

In [ ]:
k = 10

C_direct = kmeansPP(P, k)
obj_direct = kmeansObj(P, C_direct)

def run_experiment(data, k, k1, trials=3):
    res = []

    for i in range(trials):
        X = kcenter(data, k1)

        C = kmeansPP(X, k)

        val = kmeansObj(data, C)
        res.append(val)

    # averaging manually
    total = 0
    for v in res:
        total += v

    return total / len(res)


for k1 in [50, 100, 200, 500]:
    avg_obj = run_experiment(P, 10, k1)
    print(f"k1={k1}, avg Coreset objective={avg_obj}")
    print("Direct kmeans++ objective:", obj_direct)

k1=50, avg Coreset objective=582126.0252759869
Direct kmeans++ objective: 34261.63924443453
k1=100, avg Coreset objective=256396.22403000479
Direct kmeans++ objective: 34261.63924443453
k1=200, avg Coreset objective=99012.06361809916
Direct kmeans++ objective: 34261.63924443453
k1=500, avg Coreset objective=37183.60849166643
Direct kmeans++ objective: 34261.63924443453


As k1 increases, the coreset better approximates the original dataset, resulting in a decrease in the objective value. For sufficiently large k1, the performance of the coreset-based k-means++ approaches that of direct k-means++.

However, even for relatively large k1, the coreset-based solution may still have a higher objective value, indicating a tradeoff between computational efficiency and clustering quality.

## Part 2: Inverted Index

### Classes implementation

In [ ]:
# my set

# Custom set implementation using Python set
# Supports add, union, and intersection
class MySet:
    def __init__(self):
        self.data = set()

    def addElement(self, element):
        self.data.add(element)

    def union(self, otherSet):
        result = MySet()
        result.data = self.data.union(otherSet.data)
        return result

    def intersection(self, otherSet):
        result = MySet()
        result.data = self.data.intersection(otherSet.data)
        return result

    def getAll(self):
        return self.data

In [ ]:
A = MySet()
B = MySet()

# adding elements to the sets
A.addElement(1)
A.addElement(2)

B.addElement(2)
B.addElement(3)

# doing union
u = A.union(B)

# intersection
inter = A.intersection(B)

print("union:", u.data)
print("intersection:", inter.data)

union: {1, 2, 3}
intersection: {2}


In [ ]:
# Represents a word occurrence in a page
# Stores page reference and word index
class Position:
    def __init__(self, pageEntry, wordIndex):
        self.pageEntry = pageEntry
        self.wordIndex = wordIndex

    def getPageEntry(self):
        return self.pageEntry

    def getWordIndex(self):
        return self.wordIndex

In [ ]:
p = Position("page1", 5)

print(p.getPageEntry())   # page1
print(p.getWordIndex())   # 5

page1
5


In [ ]:
# Word entry

# Stores all positions of a word across pages
class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = []   # list of Position objects

    def addPosition(self, position):
        self.positions.append(position)

    def addPositions(self, positions):
        self.positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return self.positions

    def getTermFrequency(self):
        return len(self.positions)

In [ ]:
w = WordEntry("data")

w.addPosition(Position("page1", 1))
w.addPosition(Position("page1", 10))
w.addPosition(Position("page2", 4))

positions = w.getAllPositionsForThisWord()

for pos in positions:
    print(pos.getPageEntry(), pos.getWordIndex())

print("Frequency:", w.getTermFrequency())

page1 1
page1 10
page2 4
Frequency: 3


In [ ]:
#Page Index

# Stores word entries for a single page
# Maps word → WordEntry
class PageIndex:
    def __init__(self):
        self.wordEntries = {}  # word → WordEntry

    def addPositionForWord(self, word, position):
        if word not in self.wordEntries:
            self.wordEntries[word] = WordEntry(word)

        self.wordEntries[word].addPosition(position)

    def getWordEntries(self):
        return list(self.wordEntries.values())

In [ ]:
pi = PageIndex()

pi.addPositionForWord("data", Position("page1", 1))
pi.addPositionForWord("data", Position("page1", 10))
pi.addPositionForWord("structures", Position("page1", 2))

for entry in pi.getWordEntries():
    print("Word:", entry.word)
    for pos in entry.getAllPositionsForThisWord():
        print(" ", pos.getPageEntry(), pos.getWordIndex())

Word: data
  page1 1
  page1 10
Word: structures
  page1 2


In [ ]:
#Page Entry

# Represents one webpage
# Reads file, cleans text, removes stopwords, builds index
class PageEntry:
    def __init__(self, pageName):
        self.pageName = pageName
        self.pageIndex = PageIndex()
        self.totalWords = 0

        self.processPage()

    def processPage(self):
        # list of stop words
        stop_words = {
            "a","an","the","they","these","this",
            "for","is","are","was","of","or",
            "and","does","will","whose"
        }
        # open file
        filepath = f"Assignment 4- datasets\\Q2- webSearch\\webpages\\{self.pageName}"

        with open(filepath, 'r') as f:
            text = f.read().lower()

        # remove punctuation
        text = re.sub(r'[^\w\s]', ' ', text)

        words = text.split()

        index = 1

        for word in words:
            # convert plural to singular
            if len(word) > 3 and word.endswith('s'):
                word = word[:-1]
            # skip stopwords
            if word in stop_words:
                index += 1
                continue
            # skip short words
            if len(word) <= 2:
                index += 1
                continue
            # add word to index
            self.pageIndex.addPositionForWord(
                word,
                Position(self, index)
            )

            self.totalWords += 1
            index += 1

    def getPageIndex(self):
        return self.pageIndex

In [ ]:
name = ["stackoverflow", "references", "stack_oracle"]

for n in name:
    pe = PageEntry(n)
    entries = pe.getPageIndex().getWordEntries()

    print(f"\nPage: {n}")
    for entry in entries[:5]:
        print(entry.word, entry.getTermFrequency())


Page: stackoverflow
stack 1
overflow 1
java 1
how 1
repair 1

Page: references
stack_datastructure_wiki 1
http 6
wikipedia 1
org 1
wiki 1

Page: stack_oracle
oracle 1
javase 1
doc 1
api 1
java 1


In [ ]:
# InvertedPageIndex

# Global index: word - all pages containing it

class InvertedPageIndex:
    def __init__(self):
        self.index = {}  # word → WordEntry

    def addPage(self, pageEntry):
        # get words from page
        pageIndex = pageEntry.getPageIndex()

        for wordEntry in pageIndex.getWordEntries():
            word = wordEntry.word

            # create entry if not exists
            if word not in self.index:
                self.index[word] = WordEntry(word)

            # add positions
            self.index[word].addPositions(
                wordEntry.getAllPositionsForThisWord()
            )

    def getPagesWhichContainWord(self, word):
        word = word.lower()

        if word.endswith('s') and len(word) > 3:
            word = word[:-1]

        if word not in self.index:
            return None

        pages = set()

        for pos in self.index[word].getAllPositionsForThisWord():
            pages.add(pos.getPageEntry().pageName)

        return pages

In [ ]:
ipi = InvertedPageIndex()

pages = ["stackoverflow", "references", "stack_oracle"]

for p in pages:
    pe = PageEntry(p)
    ipi.addPage(pe)

result = ipi.getPagesWhichContainWord("stack")

print("Pages containing 'stack':", result)

Pages containing 'stack': {'references', 'stackoverflow', 'stack_oracle'}


### Query results

In [ ]:
# SearchEngine

# 1. Adding pages to the system
# 2. Searching which pages contain a word
# 3. Finding positions of a word in a page
class SearchEngine:
    def __init__(self):
        self.invertedIndex = InvertedPageIndex()#stores word to pages mapping
        self.pages = {}

    # Adds a new page to the search engine
    # Input: pageName (string)
    def addPage(self, pageName):
        pe = PageEntry(pageName)
        self.pages[pageName] = pe #store page in dictionary
        self.invertedIndex.addPage(pe) #add page to inverted index

    # Finds all pages that contain a given word
    # Input: word (string)
    def queryFindPagesWhichContainWord(self, word):
        result = self.invertedIndex.getPagesWhichContainWord(word) # get pages from inverted index

        if not result:
            print(f"No webpage contains word {word}")
        else:
            print(", ".join(result))

    # Finds positions of a word in a specific page
    # Input: word (string), pageName (string)
    def queryFindPositionsOfWordInAPage(self, word, pageName):
        if pageName not in self.pages: #check if page exists
            print(f"No webpage {pageName} found")
            return

        pe = self.pages[pageName] #get PageEntry and its index
        pageIndex = pe.getPageIndex()

        word = word.lower() #normalize word
        if word.endswith('s') and len(word) > 3:
            word = word[:-1]

        if word not in pageIndex.wordEntries: #check if word exists in page
            print(f"Webpage {pageName} does not contain word {word}")
            return

        positions = pageIndex.wordEntries[word].getAllPositionsForThisWord()#get all positions of word

        indices = [str(pos.getWordIndex()) for pos in positions] #extract indices from Position objects
        print(", ".join(indices))

In [ ]:
se = SearchEngine()

with open("Assignment 4- datasets\\Q2- webSearch\\actions.txt", "r") as f:
    for line in f:
        print(f"\n>>> {line.strip()}")

        parts = line.strip().split()

        if parts[0] == "addPage":
            se.addPage(parts[1])

        elif parts[0] == "queryFindPagesWhichContainWord":
            se.queryFindPagesWhichContainWord(parts[1])

        elif parts[0] == "queryFindPositionsOfWordInAPage":
            se.queryFindPositionsOfWordInAPage(parts[1], parts[2])


>>> addPage stack_datastructure_wiki

>>> queryFindPagesWhichContainWord delhi
No webpage contains word delhi

>>> queryFindPagesWhichContainWord stack
stack_datastructure_wiki

>>> queryFindPagesWhichContainWord wikipedia
stack_datastructure_wiki

>>> queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazine

>>> queryFindPagesWhichContainWord allain
No webpage contains word allain

>>> addPage stack_cprogramming

>>> queryFindPagesWhichContainWord allain
stack_cprogramming

>>> queryFindPagesWhichContainWord C
No webpage contains word C

>>> queryFindPagesWhichContainWord C++
No webpage contains word C++

>>> addPage stack_oracle

>>> queryFindPagesWhichContainWord jdk
stack_oracle

>>> addPage stackoverflow

>>> queryFindPagesWhichContainWord function
stackoverflow, stack_cprogramming, stack_datastructure_wiki

>>> addPage stacklighting

>>> addPage stackmagazine

>>> queryFindPagesWhichContainWord magazines
s

## Part 3: PageRank

### Setup

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

In [18]:
#Read file as RDD
file_path = "Assignment 4- datasets\\Q3 - PageRank\\small.txt"

edges = sc.textFile(file_path) \
    .map(lambda line: tuple(map(int, line.strip().split()))) \
    .distinct()

In [19]:
print(edges.take(5))

[(13, 1), (89, 1), (79, 1), (65, 1), (25, 1)]


In [20]:
#Build graph (outgoing links)
links = edges.groupByKey().mapValues(list)
print(links.take(5))

[(26, [2, 3, 27, 33, 65, 79, 95]), (58, [2, 6, 14, 40, 46, 48, 100, 1, 27, 41, 51, 53, 59, 87, 93]), (66, [4, 56, 51, 53, 61, 67, 71, 99]), (82, [4, 14, 22, 52, 96, 1, 7, 9, 23, 61, 83, 95]), (24, [6, 50, 66, 78, 1, 25, 53, 63, 99])]


In [21]:
#Initialize ranks
nodes = links.keys().distinct()
n = nodes.count()

ranks = nodes.map(lambda node: (node, 1.0 / n))

print(ranks.take(5))
print("Total nodes:", n)

[(26, 0.01), (58, 0.01), (66, 0.01), (82, 0.01), (24, 0.01)]
Total nodes: 100


In [22]:
#Core PageRank Iteration
beta = 0.8
iterations = 40

for i in range(iterations):

    # Join links with ranks
    joined = links.join(ranks)

    # Compute contributions
    contribs = joined.flatMap(
        lambda x: [(dest, x[1][1] / len(x[1][0])) for dest in x[1][0]]
    )

    # Update ranks
    ranks = contribs.reduceByKey(lambda a, b: a + b) \
        .mapValues(lambda r: (1 - beta)/n + beta * r)

    if i % 10 == 0:
        print(f"Iteration {i} done")

Iteration 0 done
Iteration 10 done
Iteration 20 done
Iteration 30 done


In [24]:
ranks = ranks.cache()
ranks.count()
top1 = ranks.sortBy(lambda x: -x[1]).take(1)
print(top1)

[(53, 0.03573120223267161)]


In [6]:
#Read file as RDD
file_path = "Assignment 4- datasets\\Q3 - PageRank\\whole.txt"

edges = sc.textFile(file_path) \
    .map(lambda line: tuple(map(int, line.strip().split()))) \
    .distinct()

In [7]:
print(edges.take(5))

[(994, 918), (367, 281), (816, 774), (641, 231), (322, 46)]


In [9]:
#Build graph (outgoing links)
links = edges.groupByKey().mapValues(list)
print(links.take(5))

[(994, [918, 240, 540, 995, 665, 187, 163, 943]), (816, [774, 208, 894, 762, 817, 999, 849]), (322, [46, 116, 794, 406, 816, 323, 969, 883, 27]), (400, [840, 170, 740, 738, 314, 401, 689, 631, 77]), (152, [792, 950, 688, 153, 351, 929, 459, 945, 797, 859])]


In [10]:
#Initialize ranks
nodes = links.keys().distinct()
n = nodes.count()

ranks = nodes.map(lambda node: (node, 1.0 / n))

print(ranks.take(5))
print("Total nodes:", n)

[(994, 0.001), (816, 0.001), (322, 0.001), (400, 0.001), (152, 0.001)]
Total nodes: 1000


In [11]:
#Core PageRank Iteration
beta = 0.8
iterations = 40

for i in range(iterations):

    # Join links with ranks
    joined = links.join(ranks)

    # Compute contributions
    contribs = joined.flatMap(
        lambda x: [(dest, x[1][1] / len(x[1][0])) for dest in x[1][0]]
    )

    # Update ranks
    ranks = contribs.reduceByKey(lambda a, b: a + b) \
        .mapValues(lambda r: (1 - beta)/n + beta * r)

    if i % 10 == 0:
        print(f"Iteration {i} done")

Iteration 0 done
Iteration 10 done
Iteration 20 done
Iteration 30 done


In [17]:
ranks = ranks.cache()
ranks.count()
top5 = ranks.sortBy(lambda x: -x[1]).take(5)
bottom5 = ranks.sortBy(lambda x: x[1]).take(5)

print("Top 5 nodes:", top5)
print("Bottom 5 nodes:", bottom5)

Top 5 nodes: [(263, 0.002020291181518219), (537, 0.0019433415714531492), (965, 0.0019254478071662631), (243, 0.001852634016241731), (285, 0.0018273721700645144)]
Bottom 5 nodes: [(558, 0.0003286018525215297), (93, 0.0003513568937516577), (62, 0.00035314810510596274), (424, 0.00035481538649301454), (408, 0.00038779848719291705)]
